In [1]:
# %%
# Cell 0. Discover all generated size-16 cases from earlier runs, for all generated case/model folders

from pathlib import Path

ROOT = Path("..").resolve()

MODEL_SPECS = [
    {
        "model_label": "resnet50_geirhos_tl",
        "onnx_path": ROOT / "data" / "model_midlayer" / "resnet50_geirhos_tl_with_midmaps.onnx",
    },
    {
        "model_label": "resnet50_tl_20250829",
        "onnx_path": ROOT / "data" / "model_midlayer" / "resnet50_tl_20250829_with_midmaps.onnx",
    },
]

MODEL_SPEC_BY_LABEL = {m["model_label"]: m for m in MODEL_SPECS}

for spec in MODEL_SPECS:
    assert spec["onnx_path"].exists(), f"Missing ONNX model: {spec['onnx_path']}"

print("Available evaluation models:", sorted(MODEL_SPEC_BY_LABEL))

CLASS_NAMES_JSONL = ROOT / "data" / "class_names.jsonl"
assert CLASS_NAMES_JSONL.exists(), f"Missing class_names.jsonl: {CLASS_NAMES_JSONL}"

for spec in MODEL_SPECS:
    assert spec["onnx_path"].exists(), f"Missing ONNX model: {spec['onnx_path']}"

GENERATED_ROOT = ROOT / "data" / "cases" / "generated_from_best_occluders"
OUT_DIR = ROOT / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 8

def discover_generated_cases(root: Path, target_size: int = 16):
    cases = []

    for case_dir in sorted(root.iterdir()):
        if not case_dir.is_dir():
            continue

        case_id = case_dir.name

        for source_model_dir in sorted(case_dir.iterdir()):
            if not source_model_dir.is_dir():
                continue

            source_model_type = source_model_dir.name
            size_dir = source_model_dir / f"size_{target_size}"
            gen_dir = size_dir / "generated"

            paths = {
                "case_id": case_id,
                "source_model_type": source_model_type,
                "occluder_size": target_size,
                "case_dir": size_dir,
                "gen_dir": gen_dir,
                "source_config_json": gen_dir / "source_occluder_config.json",
                "occluded_png": size_dir / "occluded.png",
                "gt_png": size_dir / "gt.png",
                "shapes_xy_npz": gen_dir / "shapes_xy.npz",
                "completions_dir": gen_dir / "completions",
                "shapes_meta_jsonl": gen_dir / "shapes_meta.jsonl",
            }

            required = [
                "source_config_json",
                "occluded_png",
                "gt_png",
                "shapes_xy_npz",
                "completions_dir",
            ]

            ok = True
            for k in required:
                if not paths[k].exists():
                    print(f"Missing {k} for {case_id} | {source_model_type}:", paths[k])
                    ok = False

            if ok:
                cases.append(paths)

    return cases


cases = discover_generated_cases(GENERATED_ROOT, target_size=TARGET_SIZE)

print("Generated root:", GENERATED_ROOT)
print("Target size   :", TARGET_SIZE)
print("Cases ready   :", len(cases))
print("Source model folders found:", sorted({c['source_model_type'] for c in cases}))
print("Processing ONNX models    :", [m["model_label"] for m in MODEL_SPECS])

Available evaluation models: ['resnet50_geirhos_tl', 'resnet50_tl_20250829']
Missing source_config_json for ns_dog_780 | resnet50_tl_20250829: /data/storage-occ-v2/repos/monte-carlo-selection/data/cases/generated_from_best_occluders/ns_dog_780/resnet50_tl_20250829/size_8/generated/source_occluder_config.json
Missing shapes_xy_npz for ns_dog_780 | resnet50_tl_20250829: /data/storage-occ-v2/repos/monte-carlo-selection/data/cases/generated_from_best_occluders/ns_dog_780/resnet50_tl_20250829/size_8/generated/shapes_xy.npz
Generated root: /data/storage-occ-v2/repos/monte-carlo-selection/data/cases/generated_from_best_occluders
Target size   : 8
Cases ready   : 79
Source model folders found: ['resnet50_geirhos_tl', 'resnet50_tl_20250829']
Processing ONNX models    : ['resnet50_geirhos_tl', 'resnet50_tl_20250829']


In [2]:
# %%
# Cell 1. Load class names

import json

with CLASS_NAMES_JSONL.open("r", encoding="utf-8") as f:
    classes = [json.loads(line)["class_name"] for line in f if line.strip()]

print("Loaded class list:", len(classes))
print("First 10 classes:", classes[:10])

Loaded class list: 54
First 10 classes: ['ant', 'bat', 'bear', 'bee', 'beetle', 'bird', 'bug', 'bull', 'butterfly', 'camel']


In [3]:
# %%
# Cell. Build ONNX sessions

import onnxruntime as ort
import numpy as np
from PIL import Image
from pathlib import Path

IM_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_png(p: str | Path) -> np.ndarray:
    img = Image.open(p).convert("RGB").resize((IM_SIZE, IM_SIZE), resample=Image.BILINEAR)
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = (x - MEAN[None, None, :]) / STD[None, None, :]
    x = np.transpose(x, (2, 0, 1))[None, ...]
    return x.astype(np.float32, copy=False)

def softmax(logits: np.ndarray) -> np.ndarray:
    z = np.asarray(logits, dtype=np.float64).reshape(-1)
    z = z - np.max(z)
    e = np.exp(z)
    return e / (np.sum(e) + 1e-12)

SESSIONS = {}

for spec in MODEL_SPECS:
    so = ort.SessionOptions()
    so.intra_op_num_threads = 30
    so.inter_op_num_threads = 1

    session = ort.InferenceSession(
        str(spec["onnx_path"]),
        sess_options=so,
        providers=["CPUExecutionProvider"],
    )

    outs = [o.name for o in session.get_outputs()]
    assert "logits" in outs, f"{spec['model_label']}: missing logits"
    assert "layer2_map" in outs, f"{spec['model_label']}: missing layer2_map"
    assert "layer3_map" in outs, f"{spec['model_label']}: missing layer3_map"

    SESSIONS[spec["model_label"]] = {
        "session": session,
        "input_name": session.get_inputs()[0].name,
        "logits_name": "logits",
        "l2_name": "layer2_map",
        "l3_name": "layer3_map",
        "onnx_path": spec["onnx_path"],
    }

print("Loaded sessions:", list(SESSIONS.keys()))

Loaded sessions: ['resnet50_geirhos_tl', 'resnet50_tl_20250829']


In [4]:
# %%
# Cell 3. Model-aware inference helpers

def infer_maps_and_logits(p: str | Path, model_label: str):
    sess_info = SESSIONS[model_label]
    x = preprocess_png(p)
    l2, l3, logits = sess_info["session"].run(
        [sess_info["l2_name"], sess_info["l3_name"], sess_info["logits_name"]],
        {sess_info["input_name"]: x}
    )
    return np.asarray(l2), np.asarray(l3), np.asarray(logits).reshape(-1)

def target_from_occluded(case: dict, model_label: str):
    _, _, occ_logits = infer_maps_and_logits(case["occluded_png"], model_label=model_label)
    occ_prob = softmax(occ_logits)
    target_idx = int(np.argmax(occ_logits))
    return target_idx, float(occ_logits[target_idx]), float(occ_prob[target_idx])

In [5]:
# %%
# Cell 4. Load shapes_xy.npz for a case and resolve completion PNG paths

import numpy as np
from pathlib import Path

def load_case_shapes(case: dict):
    z = np.load(case["shapes_xy_npz"], allow_pickle=True)
    out_files_raw = z["out_files"].tolist()
    polygons_xy   = z["polygons"]
    matlab_1_indexed = bool(z["matlab_1_indexed"]) if "matlab_1_indexed" in z else False

    completions_dir = case["completions_dir"]

    def resolve_png_path(p: str) -> Path:
        pth = Path(p)
        if pth.exists():
            return pth
        return completions_dir / pth.name

    png_paths = [resolve_png_path(p) for p in out_files_raw]
    missing = sum(1 for p in png_paths if not p.exists())

    return png_paths, polygons_xy, matlab_1_indexed, missing

In [6]:
# %%
# Cell 6. Model-aware parallel scoring

from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
from tqdm.auto import tqdm

N_WORKERS = 60

def _score_chunk_onnx(args):
    chunk_pairs, target_idx, occ_tlog, onnx_path_str, logits_out_name = args

    import os
    import numpy as np
    import onnxruntime as ort
    from PIL import Image

    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    so.inter_op_num_threads = 1

    sess = ort.InferenceSession(onnx_path_str, sess_options=so, providers=["CPUExecutionProvider"])
    in_name = sess.get_inputs()[0].name

    IM_SIZE = 224
    MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def preprocess(p):
        img = Image.open(p).convert("RGB").resize((IM_SIZE, IM_SIZE), resample=Image.BILINEAR)
        x = np.asarray(img, dtype=np.float32) / 255.0
        x = (x - MEAN[None, None, :]) / STD[None, None, :]
        x = np.transpose(x, (2, 0, 1))[None, ...]
        return x.astype(np.float32, copy=False)

    def softmax_local(logits):
        z = np.asarray(logits, dtype=np.float64).reshape(-1)
        z = z - np.max(z)
        e = np.exp(z)
        return e / (np.sum(e) + 1e-12)

    idxs = []
    v_tlog = []
    v_tpr  = []
    v_tmar = []
    v_tdel = []

    for i, pstr in chunk_pairs:
        try:
            if not os.path.exists(pstr):
                continue

            x = preprocess(pstr)
            logits = sess.run([logits_out_name], {in_name: x})[0]
            logits = np.asarray(logits).reshape(-1)

            prob = softmax_local(logits)
            tl = float(logits[int(target_idx)])
            other_max = float(np.max(np.delete(logits, int(target_idx))))

            idxs.append(int(i))
            v_tlog.append(tl)
            v_tpr.append(float(prob[int(target_idx)]))
            v_tdel.append(float(tl - float(occ_tlog)))
            v_tmar.append(float(tl - other_max))
        except Exception:
            continue

    return (
        np.asarray(idxs, dtype=np.int64),
        np.asarray(v_tlog, dtype=np.float64),
        np.asarray(v_tpr, dtype=np.float64),
        np.asarray(v_tmar, dtype=np.float64),
        np.asarray(v_tdel, dtype=np.float64),
        int(len(idxs)),
    )


def score_case_parallel(case: dict, png_paths: list, target_idx: int, occ_tlog: float, model_label: str):
    N = len(png_paths)
    tlog = np.full(N, np.nan, dtype=np.float64)
    tpr  = np.full(N, np.nan, dtype=np.float64)
    tmar = np.full(N, np.nan, dtype=np.float64)
    tdel = np.full(N, np.nan, dtype=np.float64)

    pairs = [(i, str(p)) for i, p in enumerate(png_paths)]
    if len(pairs) == 0:
        print("No png_paths for case:", case.get("case_id", "unknown"))
        return tlog, tpr, tmar, tdel

    sess_info = SESSIONS[model_label]
    onnx_path = str(sess_info["onnx_path"])
    logits_name = sess_info["logits_name"]

    chunk_size = max(1, len(pairs) // (N_WORKERS * 30))
    chunks = [pairs[j:j+chunk_size] for j in range(0, len(pairs), chunk_size)]
    args_list = [(ch, target_idx, occ_tlog, onnx_path, logits_name) for ch in chunks]

    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_score_chunk_onnx, a) for a in args_list]
        for fut in tqdm(as_completed(futs), total=len(futs), desc=f"Scoring {case.get('case_id','case')} | {model_label}"):
            idxs, a, b, c, d, _ = fut.result()
            if idxs.size:
                tlog[idxs] = a
                tpr[idxs]  = b
                tmar[idxs] = c
                tdel[idxs] = d

    return tlog, tpr, tmar, tdel

In [7]:
# %%
# Cell 7. New occluder mask builder from source_occluder_config.json

import json
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

def build_occluder_mask_from_source_config(case: dict):
    cfg = json.loads(Path(case["source_config_json"]).read_text(encoding="utf-8"))

    x0 = float(cfg["x0_norm"])
    y0 = float(cfg["y0_norm"])
    x1 = float(cfg["x1_norm"])
    y1 = float(cfg["y1_norm"])

    W_img, H_img = Image.open(case["occluded_png"]).size

    occluder_px = np.array([
        [x0 * (W_img - 1), y0 * (H_img - 1)],
        [x1 * (W_img - 1), y0 * (H_img - 1)],
        [x1 * (W_img - 1), y1 * (H_img - 1)],
        [x0 * (W_img - 1), y1 * (H_img - 1)],
    ], dtype=np.float64)

    occ_mask_img = Image.new("L", (W_img, H_img), 0)
    draw = ImageDraw.Draw(occ_mask_img)
    draw.polygon([(float(x), float(y)) for x, y in occluder_px], outline=1, fill=1)
    occluder_mask = (np.asarray(occ_mask_img, dtype=np.uint8) > 0)

    return occluder_mask, occluder_px, (W_img, H_img)

In [8]:
# %%
# Cell 8. Batch BGMM output base

from pathlib import Path
import json

BGMM_DIR = OUT_DIR / "occlusion" / "bgmm_generated_size16_both_models"
BGMM_DIR.mkdir(parents=True, exist_ok=True)

def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("BGMM_DIR:", BGMM_DIR)

BGMM_DIR: /data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models


In [9]:
# %%
# Cell 9. Imports used by the single-case-equivalent pipeline

import numpy as np
import pandas as pd
from datetime import datetime, timezone
from PIL import Image, ImageDraw
from shapely.geometry import Point, Polygon
from sklearn.decomposition import PCA
from sklearn.mixture import BayesianGaussianMixture
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

In [10]:
# %%
# Cell 10. Core pipeline functions

def build_eligible_set(tlog: np.ndarray, *, valid_poly=None) -> np.ndarray:
    N_total = int(tlog.shape[0])
    E = np.isfinite(tlog)

    if valid_poly is not None:
        vp = np.asarray(valid_poly, dtype=bool)
        if vp.shape[0] == N_total:
            E = E & vp
        else:
            print("Warning: valid_poly length mismatch. Skipping valid_poly filter.")

    return np.where(E)[0].astype(int)


def run_pca(Xn: np.ndarray, PCA_D: int = 30):
    pca = PCA(n_components=int(PCA_D), random_state=0)
    Z = pca.fit_transform(Xn)
    evr = np.asarray(pca.explained_variance_ratio_, dtype=np.float64)
    return Z, pca, evr


def run_bgmm(Z: np.ndarray, K_MAX: int = 80, ACTIVE_THR: float = 1e-4):
    bgmm = BayesianGaussianMixture(
        n_components=int(K_MAX),
        covariance_type="diag",
        weight_concentration_prior_type="dirichlet_process",
        weight_concentration_prior=1e-2,
        max_iter=2000,
        random_state=0,
        init_params="kmeans",
        reg_covar=1e-6,
    )
    bgmm.fit(Z)

    resp = bgmm.predict_proba(Z)
    mix_w = np.asarray(bgmm.weights_, dtype=np.float64)
    active = np.where(mix_w > float(ACTIVE_THR))[0].astype(int)
    return bgmm, resp, mix_w, active


def turning_curvature(polyline_xy: np.ndarray, eps: float = 1e-12) -> float:
    x = np.asarray(polyline_xy, dtype=np.float64)
    if x.shape[0] < 3:
        return np.nan
    v1 = x[1:-1] - x[:-2]
    v2 = x[2:]   - x[1:-1]
    n1 = np.linalg.norm(v1, axis=1) + eps
    n2 = np.linalg.norm(v2, axis=1) + eps
    v1u = v1 / n1[:, None]
    v2u = v2 / n2[:, None]
    dot = np.clip(np.sum(v1u * v2u, axis=1), -1.0, 1.0)
    cross = v1u[:, 0] * v2u[:, 1] - v1u[:, 1] * v2u[:, 0]
    ang = np.arctan2(cross, dot)
    total_turn = np.sum(np.abs(ang))
    seg = x[1:] - x[:-1]
    arc_len = float(np.sum(np.linalg.norm(seg, axis=1)))
    return float(total_turn / (arc_len + eps))


def _points_inside_occluder_xy(xy: np.ndarray, occ_poly_wkb: bytes) -> np.ndarray:
    from shapely import wkb
    from shapely.geometry import Point
    poly = wkb.loads(occ_poly_wkb)

    if xy.size == 0:
        return xy.reshape(0, 2)
    inside = []
    for p in xy:
        if poly.contains(Point(float(p[0]), float(p[1]))):
            inside.append(p)
    return np.asarray(inside, dtype=np.float64) if inside else np.zeros((0, 2), dtype=np.float64)


def _curv_one(args):
    j, i, pts, occ_poly_wkb, m1 = args
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    if m1:
        pts = pts - 1.0
    pts_in = _points_inside_occluder_xy(pts, occ_poly_wkb)
    if pts_in.shape[0] >= 3:
        c = turning_curvature(pts_in)
    else:
        c = turning_curvature(pts)
    return j, float(c)


def compute_curvatures(polygons_xy, E_idx, occluder_px, matlab_1_indexed_flag: bool, N_WORKERS: int = 60):
    occ_poly = Polygon(np.asarray(occluder_px, dtype=np.float64).tolist())
    occ_wkb = occ_poly.wkb

    tasks = [
        (j, int(i), polygons_xy[int(i)], occ_wkb, bool(matlab_1_indexed_flag))
        for j, i in enumerate(E_idx)
    ]

    curv = np.full((len(E_idx),), np.nan, dtype=np.float64)

    with ProcessPoolExecutor(max_workers=int(N_WORKERS)) as ex:
        futs = [ex.submit(_curv_one, t) for t in tasks]
        for fut in as_completed(futs):
            j, c = fut.result()
            curv[j] = c

    return curv


def _png_to_shape_mask_from_path(p_str: str) -> np.ndarray:
    arr = np.asarray(Image.open(p_str).convert("L"), dtype=np.uint8)
    return arr < 128


def _worker_chunk_fill(args):
    js, idxs, png_paths_str, occluder_mask, baseline_mask = args

    fill1 = np.full((len(js),), np.nan, dtype=np.float64)
    fill2 = np.full((len(js),), np.nan, dtype=np.float64)

    from pathlib import Path as _Path

    for k, (j, i) in enumerate(zip(js, idxs)):
        p = png_paths_str[int(i)]
        if not p:
            continue
        if not _Path(p).exists():
            continue

        m = _png_to_shape_mask_from_path(p)
        fill1[k] = float(m[occluder_mask].mean())

        added = m & (~baseline_mask)
        fill2[k] = float(added[occluder_mask].mean())

    return js, fill1, fill2


def compute_fill_metrics(
    png_paths,
    E_idx,
    occluder_mask,
    occluded_png_path,
    N_WORKERS: int = 30,
    CHUNK_SIZE: int = 150,
):
    baseline_mask = _png_to_shape_mask_from_path(str(occluded_png_path))
    png_paths_str = [str(p) for p in png_paths]

    fill_in_occ = np.full((len(E_idx),), np.nan, dtype=np.float64)
    fill_added_in_occ = np.full((len(E_idx),), np.nan, dtype=np.float64)

    chunks = []
    for start in range(0, len(E_idx), int(CHUNK_SIZE)):
        stop = min(len(E_idx), start + int(CHUNK_SIZE))
        js = list(range(start, stop))
        idxs = [int(E_idx[j]) for j in js]
        chunks.append((js, idxs, png_paths_str, occluder_mask, baseline_mask))

    with ProcessPoolExecutor(max_workers=int(N_WORKERS)) as ex:
        futs = [ex.submit(_worker_chunk_fill, c) for c in chunks]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="fill metrics"):
            js, fill1, fill2 = fut.result()
            fill_in_occ[js] = fill1
            fill_added_in_occ[js] = fill2

    return fill_in_occ, fill_added_in_occ


def invfreq_weights_quantile(x: np.ndarray, bins: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    qs = np.linspace(0.0, 1.0, int(bins) + 1)
    edges = np.quantile(x, qs)
    edges[0] -= 1e-12
    edges[-1] += 1e-12
    b = np.digitize(x, edges) - 1
    b = np.clip(b, 0, bins - 1)
    cnt = np.bincount(b, minlength=bins).astype(np.float64)
    p = cnt / (cnt.sum() + 1e-12)
    w = 1.0 / (p[b] + 1e-12)
    w = w / (w.mean() + 1e-12)
    return w


def compute_debias_weights(curv, fill_in_occ, *, BINS_CURV=20, BINS_SIZE=20, CAP=10.0):
    size_metric = np.asarray(fill_in_occ, dtype=np.float64)

    curv_safe = np.asarray(curv, dtype=np.float64).copy()
    curv_safe[~np.isfinite(curv_safe)] = float(np.nanmedian(curv_safe))

    ok = np.isfinite(curv_safe) & np.isfinite(size_metric)
    if int(ok.sum()) < 50:
        raise RuntimeError(f"Too few valid samples for debiasing: {int(ok.sum())}")

    w_curv = np.ones((len(curv_safe),), dtype=np.float64)
    w_size = np.ones((len(curv_safe),), dtype=np.float64)

    w_curv[ok] = invfreq_weights_quantile(curv_safe[ok], bins=int(BINS_CURV))
    w_size[ok] = invfreq_weights_quantile(size_metric[ok], bins=int(BINS_SIZE))

    w_joint = w_curv * w_size
    w_joint[ok] = np.clip(w_joint[ok], 1.0 / float(CAP), float(CAP))
    w_joint[ok] = w_joint[ok] / (np.mean(w_joint[ok]) + 1e-12)

    return w_joint, ok


def build_component_table(resp, mix_w, w_joint, ACTIVE_THR=1e-4):
    resp = np.asarray(resp, dtype=np.float64)
    mix_w = np.asarray(mix_w, dtype=np.float64)
    w_joint = np.asarray(w_joint, dtype=np.float64)

    mass_raw = resp.sum(axis=0)
    mass_debias = (w_joint[:, None] * resp).sum(axis=0)

    comp_df = pd.DataFrame({
        "k": np.arange(resp.shape[1], dtype=int),
        "mix_weight": mix_w,
        "mass_raw": mass_raw,
        "mass_debias": mass_debias,
    })

    active_df = comp_df.query(f"mix_weight > {float(ACTIVE_THR)}").copy()
    active_df = active_df.sort_values("mass_debias", ascending=False).reset_index(drop=True)
    return comp_df, active_df


def export_bgmm_jsonls(
    case_id: str,
    out_base_dir,
    *,
    png_paths,
    E_idx,
    Z,
    evr,
    pca_obj,
    bgmm_obj,
    resp,
    mix_w,
    comp_df,
    active_df,
    top_clusters_n: int = 10,
    top_shapes_per_cluster: int = 25,
    ACTIVE_THR: float = 1e-4,
):
    out_base_dir = Path(out_base_dir) / case_id
    out_base_dir.mkdir(parents=True, exist_ok=True)

    ts = datetime.now(timezone.utc).isoformat(timespec="seconds")

    active_order = active_df["k"].astype(int).tolist()
    k_to_rank = {int(k): int(r + 1) for r, k in enumerate(active_order)}

    meta = {
        "timestamp_utc": ts,
        "case_id": case_id,
        "n_total_completions": int(len(png_paths)),
        "n_eligible": int(len(E_idx)),
        "pca_d": int(Z.shape[1]),
        "pca_cum_explained_var": float(np.sum(evr)),
        "pca_explained_var_first10": [float(x) for x in evr[:10]],
        "bgmm_k_max": int(resp.shape[1]),
        "bgmm_active_thr": float(ACTIVE_THR),
        "n_active": int(active_df.shape[0]),
        "top_clusters_n": int(top_clusters_n),
        "top_shapes_per_cluster": int(top_shapes_per_cluster),
    }
    write_jsonl(out_base_dir / "meta.jsonl", [meta])

    comp_rows = []
    for _, row in comp_df.iterrows():
        k = int(row["k"])
        comp_rows.append({
            "timestamp_utc": ts,
            "case_id": case_id,
            "k": k,
            "mix_weight": float(row["mix_weight"]),
            "mass_raw": float(row["mass_raw"]),
            "mass_debias": float(row["mass_debias"]),
            "is_active": bool(float(row["mix_weight"]) > float(ACTIVE_THR)),
            "active_rank_by_mass_debias": int(k_to_rank[k]) if k in k_to_rank else None,
        })
    write_jsonl(out_base_dir / "components.jsonl", comp_rows)

    assign_rows = []
    resp = np.asarray(resp, dtype=np.float64)
    for j, i in enumerate(E_idx):
        r = resp[j]
        k_hat = int(np.argmax(r))
        topk = np.argsort(-r)[: min(5, r.size)].astype(int)
        assign_rows.append({
            "timestamp_utc": ts,
            "case_id": case_id,
            "eligible_row": int(j),
            "global_index": int(i),
            "png": str(png_paths[int(i)]),
            "k_argmax": k_hat,
            "k_rank_among_active": int(k_to_rank[k_hat]) if k_hat in k_to_rank else None,
            "resp_top5": [{"k": int(kk), "p": float(r[int(kk)])} for kk in topk],
        })
    write_jsonl(out_base_dir / "shape_to_cluster.jsonl", assign_rows)

    top_rows = []
    top_ks = active_df.head(int(top_clusters_n))["k"].astype(int).tolist()
    for rank, k in enumerate(top_ks, start=1):
        r_k = resp[:, int(k)]
        topj = np.argsort(-r_k)[: int(top_shapes_per_cluster)].astype(int)
        shapes = []
        for jj in topj:
            gi = int(E_idx[int(jj)])
            shapes.append({
                "eligible_row": int(jj),
                "global_index": gi,
                "png": str(png_paths[gi]),
                "resp": float(r_k[int(jj)]),
            })
        top_rows.append({
            "timestamp_utc": ts,
            "case_id": case_id,
            "cluster_rank": int(rank),
            "k": int(k),
            "mix_weight": float(active_df.loc[active_df["k"] == k, "mix_weight"].iloc[0]),
            "mass_debias": float(active_df.loc[active_df["k"] == k, "mass_debias"].iloc[0]),
            "shapes": shapes,
        })
    write_jsonl(out_base_dir / "top10_clusters.jsonl", top_rows)

    params = {
        "timestamp_utc": ts,
        "case_id": case_id,
        "bgmm_weights": [float(x) for x in np.asarray(bgmm_obj.weights_, dtype=np.float64).tolist()],
        "bgmm_means": np.asarray(bgmm_obj.means_, dtype=np.float64).tolist(),
        "bgmm_covariances": np.asarray(bgmm_obj.covariances_, dtype=np.float64).tolist(),
        "bgmm_precisions_cholesky": np.asarray(bgmm_obj.precisions_cholesky_, dtype=np.float64).tolist(),
        "pca_components": np.asarray(pca_obj.components_, dtype=np.float64).tolist(),
        "pca_mean": np.asarray(pca_obj.mean_, dtype=np.float64).tolist(),
        "pca_explained_variance": np.asarray(pca_obj.explained_variance_, dtype=np.float64).tolist(),
    }
    write_jsonl(out_base_dir / "model_params.jsonl", [params])

    return out_base_dir

In [11]:
# %%
# Cell. Model-aware ROI-pooled mid-level embeddings

import numpy as np
from PIL import Image

def build_embeddings_Xn(png_paths, E_idx, occluder_mask_imgspace, model_label: str, use_contrast: bool = True):
    """
    ROI-pooled mid-level embeddings using layer2_map + layer3_map.
    Pools activations inside the occluder region, optionally using inside-minus-outside contrast.
    Returns an L2-normalized embedding matrix Xn with shape (|E|, 512+1024).
    """

    def _resize_mask(mask_img: np.ndarray, size_hw):
        H, W = int(size_hw[0]), int(size_hw[1])
        m = Image.fromarray(mask_img.astype(np.uint8) * 255)
        m = m.resize((W, H), resample=Image.NEAREST)
        return (np.asarray(m) > 0)

    def _roi_pool_map(feat_map: np.ndarray, mask_small: np.ndarray, contrast: bool):
        fmap = np.asarray(feat_map)[0]   # (C,H,W)

        inside = mask_small[None, :, :]
        outside = ~inside
        eps = 1e-6

        mean_in = (fmap * inside).sum(axis=(1, 2)) / (inside.sum(axis=(1, 2)) + eps)

        if not contrast:
            return mean_in

        mean_out = (fmap * outside).sum(axis=(1, 2)) / (outside.sum(axis=(1, 2)) + eps)
        return mean_in - mean_out

    # infer shapes from first eligible sample
    l2, l3, _ = infer_maps_and_logits(png_paths[int(E_idx[0])], model_label=model_label)

    mask_l2 = _resize_mask(occluder_mask_imgspace, l2.shape[-2:])
    mask_l3 = _resize_mask(occluder_mask_imgspace, l3.shape[-2:])

    dim = int(l2.shape[1]) + int(l3.shape[1])   # 512 + 1024
    X = np.zeros((len(E_idx), dim), dtype=np.float32)

    for j, i in enumerate(E_idx):
        l2, l3, _ = infer_maps_and_logits(png_paths[int(i)], model_label=model_label)
        v2 = _roi_pool_map(l2, mask_l2, contrast=use_contrast)
        v3 = _roi_pool_map(l3, mask_l3, contrast=use_contrast)
        X[j] = np.concatenate([v2, v3]).astype(np.float32, copy=False)

    norm = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    return X / norm

In [12]:
# %%
# Cell. Output path helpers for source-only evaluation

from pathlib import Path

def run_output_dir(case: dict) -> Path:
    return (
        BGMM_DIR
        / f"source_{case['source_model_type']}"
        / f"size_{case['occluder_size']}"
        / case["case_id"]
    )



In [13]:
def run_already_processed(case: dict, require_all: bool = True):
    out_dir = run_output_dir(case)

    required_files = [
        out_dir / "meta.jsonl",
        out_dir / "components.jsonl",
        out_dir / "shape_to_cluster.jsonl",
        out_dir / "top10_clusters.jsonl",
        out_dir / "model_params.jsonl",
    ]

    exists = [p.exists() for p in required_files]

    if require_all:
        done = all(exists)
    else:
        done = any(exists)

    return done, out_dir, required_files

In [14]:
# %%
# Cell. Preview runs

pending_rows = []
done_rows = []

for case in cases:
    done, out_dir, required_files = run_already_processed(case, require_all=True)

    row = {
        "case_id": case["case_id"],
        "source_model_type": case["source_model_type"],
        "occluder_size": case["occluder_size"],
        "out_dir": str(out_dir),
        "already_processed": done,
    }

    if done:
        done_rows.append(row)
    else:
        pending_rows.append(row)

done_df = pd.DataFrame(done_rows)
pending_df = pd.DataFrame(pending_rows)

print("Already processed:", len(done_df))
print("Pending         :", len(pending_df))

if len(done_df):
    display(done_df.head(20))
if len(pending_df):
    display(pending_df.head(20))

Already processed: 27
Pending         : 52


,case_id,source_model_type,occluder_size,out_dir,already_processed
0,ns_cow_202,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
1,ns_cow_202,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
2,ns_cow_205,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
3,ns_cow_205,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
4,ns_cow_350,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
5,ns_cow_350,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
6,ns_cow_510,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
7,ns_cow_510,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
8,ns_cow_780,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True
9,ns_cow_780,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,True


,case_id,source_model_type,occluder_size,out_dir,already_processed
0,ns_elephant_780,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
1,ns_elephant_780,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
2,ns_lion_350,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
3,ns_lion_350,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
4,ns_lion_560,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
5,ns_lion_560,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
6,ns_lion_650,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
7,ns_lion_650,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
8,ns_lion_780,resnet50_geirhos_tl,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False
9,ns_lion_780,resnet50_tl_20250829,8,/data/storage-occ-v2/repos/monte-carlo-selecti...,False


In [15]:
# %%
# Cell. Batch run: each source model evaluates only the shapes it generated

PCA_D = 30
K_MAX = 80
ACTIVE_THR = 1e-4

N_WORKERS = 60
FILL_CHUNK = 150

TOP_CLUSTERS_N = 10
TOP_SHAPES_PER_CLUSTER = 25

export_dirs = []
skipped_runs = []

for case in cases:
    case_id = case["case_id"]
    source_model_type = case["source_model_type"]
    eval_model_label = source_model_type

    if eval_model_label not in MODEL_SPEC_BY_LABEL:
        print(f"Skipping {case_id}: no ONNX mapping found for source model '{source_model_type}'")
        continue

    done, out_dir, required_files = run_already_processed(case, require_all=True)

    if done:
        print(
            "Skipping already processed:",
            case["case_id"],
            "| source/eval =", source_model_type,
            "| size =", case["occluder_size"],
        )
        skipped_runs.append({
            "case_id": case["case_id"],
            "source_model_type": source_model_type,
            "occluder_size": case["occluder_size"],
            "out_dir": str(out_dir),
        })
        continue

    png_paths, polygons_xy, matlab_1_indexed, missing = load_case_shapes(case)
    if missing > 0:
        print("Warning:", case_id, "|", source_model_type, "missing PNGs:", missing)

    target_idx, occ_tlog, occ_tpr = target_from_occluded(case, model_label=eval_model_label)

    tlog, tpr, tmar, tdel = score_case_parallel(
        case=case,
        png_paths=png_paths,
        target_idx=target_idx,
        occ_tlog=occ_tlog,
        model_label=eval_model_label,
    )

    E_idx = build_eligible_set(tlog)

    print("\nCASE:", case_id)
    print("  source model      :", source_model_type)
    print("  eval model        :", eval_model_label)
    print("  occluder size     :", case["occluder_size"])
    print("  Eligible          :", int(len(E_idx)), "/", int(len(png_paths)))
    print(
        "  Target            :",
        classes[target_idx],
        "idx:",
        int(target_idx),
        "occ_tlog:",
        float(occ_tlog),
        "occ_tpr:",
        float(occ_tpr),
    )

    if len(E_idx) < 50:
        print("  Skipping BGMM. Too few eligible samples.")
        continue

    occluder_mask, occluder_px, _ = build_occluder_mask_from_source_config(case)

    Xn = build_embeddings_Xn(
        png_paths=png_paths,
        E_idx=E_idx,
        occluder_mask_imgspace=occluder_mask,
        model_label=eval_model_label,
        use_contrast=True,
    )

    Z, pca_obj, evr = run_pca(Xn, PCA_D=PCA_D)
    bgmm_obj, resp, mix_w, active = run_bgmm(Z, K_MAX=K_MAX, ACTIVE_THR=ACTIVE_THR)

    print("  Z:", tuple(Z.shape), "PCA cum EV:", float(np.sum(evr)))
    print("  Active components:", int(active.size), "/", int(K_MAX))

    curv = compute_curvatures(
        polygons_xy=polygons_xy,
        E_idx=E_idx,
        occluder_px=occluder_px,
        matlab_1_indexed_flag=matlab_1_indexed,
        N_WORKERS=N_WORKERS,
    )

    fill_in_occ, fill_added_in_occ = compute_fill_metrics(
        png_paths=png_paths,
        E_idx=E_idx,
        occluder_mask=occluder_mask,
        occluded_png_path=case["occluded_png"],
        N_WORKERS=N_WORKERS,
        CHUNK_SIZE=FILL_CHUNK,
    )

    w_joint, ok = compute_debias_weights(
        curv,
        fill_in_occ,
        BINS_CURV=20,
        BINS_SIZE=20,
        CAP=10.0,
    )

    comp_df, active_df = build_component_table(resp, mix_w, w_joint, ACTIVE_THR=ACTIVE_THR)

    print("  Top-5 clusters by mass_debias:")
    print(active_df.head(5)[["k", "mix_weight", "mass_debias"]])

    out_dir_case = export_bgmm_jsonls(
        case_id=case["case_id"],
        out_base_dir=(
            BGMM_DIR
            / f"source_{source_model_type}"
            / f"size_{case['occluder_size']}"
        ),
        png_paths=png_paths,
        E_idx=E_idx,
        Z=Z,
        evr=evr,
        pca_obj=pca_obj,
        bgmm_obj=bgmm_obj,
        resp=resp,
        mix_w=mix_w,
        comp_df=comp_df,
        active_df=active_df,
        top_clusters_n=TOP_CLUSTERS_N,
        top_shapes_per_cluster=TOP_SHAPES_PER_CLUSTER,
        ACTIVE_THR=ACTIVE_THR,
    )
    export_dirs.append(str(out_dir_case))

print("\nDone. Newly exported BGMM JSONLs for runs:", len(export_dirs))
for d in export_dirs[:10]:
    print("  ", d)
if len(export_dirs) > 10:
    print("  ...")

print("\nSkipped already processed runs:", len(skipped_runs))
for r in skipped_runs[:10]:
    print("  ", r["case_id"], "|", r["source_model_type"], "| size", r["occluder_size"])
if len(skipped_runs) > 10:
    print("  ...")

Skipping already processed: ns_cow_202 | source/eval = resnet50_geirhos_tl | size = 8
Skipping already processed: ns_cow_202 | source/eval = resnet50_tl_20250829 | size = 8
Skipping already processed: ns_cow_205 | source/eval = resnet50_geirhos_tl | size = 8
Skipping already processed: ns_cow_205 | source/eval = resnet50_tl_20250829 | size = 8
Skipping already processed: ns_cow_350 | source/eval = resnet50_geirhos_tl | size = 8
Skipping already processed: ns_cow_350 | source/eval = resnet50_tl_20250829 | size = 8
Skipping already processed: ns_cow_510 | source/eval = resnet50_geirhos_tl | size = 8
Skipping already processed: ns_cow_510 | source/eval = resnet50_tl_20250829 | size = 8
Skipping already processed: ns_cow_780 | source/eval = resnet50_geirhos_tl | size = 8
Skipping already processed: ns_cow_780 | source/eval = resnet50_tl_20250829 | size = 8
Skipping already processed: ns_dog_105 | source/eval = resnet50_geirhos_tl | size = 8
Skipping already processed: ns_dog_105 | source/e

Scoring ns_elephant_780 | resnet50_geirhos_tl:   0%|          | 0/2000 [00:00<?, ?it/s]


CASE: ns_elephant_780
  source model      : resnet50_geirhos_tl
  eval model        : resnet50_geirhos_tl
  occluder size     : 8
  Eligible          : 10000 / 10000
  Target            : sheep idx: 42 occ_tlog: 4.627476692199707 occ_tpr: 0.2015942161172042


  Z: (10000, 30) PCA cum EV: 0.9832615877967328
  Active components: 80 / 80


fill metrics:   0%|          | 0/67 [00:00<?, ?it/s]

...


CASE: s_oyster_4
  source model      : resnet50_tl_20250829
  eval model        : resnet50_tl_20250829
  occluder size     : 8
  Eligible          : 10000 / 10000
  Target            : dog idx: 17 occ_tlog: 117.82538604736328 occ_tpr: 0.9999999999113212


  Z: (10000, 30) PCA cum EV: 0.9880393627099693
  Active components: 80 / 80


fill metrics:   0%|          | 0/67 [00:00<?, ?it/s]

  Top-5 clusters by mass_debias:
   k  mix_weight  mass_debias
0  0    0.012672   125.734717
1  1    0.012670   125.725288
2  2    0.012668   125.715560
3  3    0.012665   125.705892
4  4    0.012663   125.696126



Done. Newly exported BGMM JSONLs for runs: 52
   /data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_8/ns_elephant_780
   /data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_tl_20250829/size_8/ns_elephant_780
   /data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_8/ns_lion_350
   /data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_tl_20250829/size_8/ns_lion_350
   /data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_geirhos_tl/size_8/ns_lion_560
   /data/storage-occ-v2/repos/monte-carlo-selection/results/occlusion/bgmm_generated_size16_both_models/source_resnet50_tl_20250829/size_8/ns_lion_560
   /data/storage-occ-v2/repos/monte-carlo-